# Attention experiments

For each experiment, write a prediction before running it. Change one variable,
record the result, and explain it. The saved text outputs were produced by
executing these cells. Calculations use Python's standard library.
Optional plots require Matplotlib; the numerical experiments still run without it.


In [1]:
from pathlib import Path
from math import isclose, log, sqrt
import runpy

candidates = [Path.cwd(), Path.cwd() / '02-transformers' / 'attention']
topic = next((p for p in candidates if (p / 'implementation.py').is_file()), None)
if topic is None:
    raise FileNotFoundError('Run from the attention folder or the AI root.')
module = runpy.run_path(str(topic / 'implementation.py'))
attention, softmax, project = (module[name] for name in ('attention', 'softmax', 'project'))
print_matrix = module['print_matrix']
X = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
tokens = ['I', 'like', 'tea']
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
print('Loaded implementation. Optional plots:', 'available' if plt else 'Matplotlib not installed')


Loaded implementation. Optional plots: Matplotlib not installed


## 1. Causal masking and future-token isolation

Predict: which outputs should change if the final token changes?
Also try the incorrect order: softmax first, then zero future weights.


In [2]:
_, unmasked_weights, unmasked_outputs = attention(X, X, X)
_, causal_weights, causal_outputs = attention(X, X, X, causal=True)
print_matrix('Unmasked weights:', unmasked_weights)
print_matrix('Causal weights:', causal_weights)
changed = [X[0][:], X[1][:], [10.0, 10.0]]
_, _, changed_causal = attention(changed, changed, changed, causal=True)
_, _, changed_unmasked = attention(changed, changed, changed)
assert changed_causal[:2] == causal_outputs[:2]
assert changed_unmasked[0] != unmasked_outputs[0]
print('Changing the final token preserves the first two causal outputs:', changed_causal[:2])
wrong = [unmasked_weights[0][0], 0.0, 0.0]
print('Incorrect mask after softmax, row sum:', round(sum(wrong), 6))
assert not isclose(sum(wrong), 1.0)
if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    for axis, weights, title in zip(axes, [unmasked_weights, causal_weights], ['Unmasked', 'Causal']):
        axis.imshow(weights, vmin=0, vmax=1, cmap='Blues')
        axis.set_xticks(range(3), tokens)
        axis.set_yticks(range(3), tokens)
        axis.set_xlabel('Key / value position')
        axis.set_ylabel('Query position')
        axis.set_title(title)
        for i in range(3):
            for j in range(3):
                axis.text(j, i, f'{weights[i][j]:.2f}', ha='center', va='center')
    plt.tight_layout()
    plt.show()


Unmasked weights:
  [0.401112, 0.197776, 0.401112]
  [0.197776, 0.401112, 0.401112]
  [0.248255, 0.248255, 0.503490]
Causal weights:
  [1.000000, 0.000000, 0.000000]
  [0.330238, 0.669762, 0.000000]
  [0.248255, 0.248255, 0.503490]
Changing the final token preserves the first two causal outputs: [[1.0, 0.0], [0.3302384506733431, 0.6697615493266569]]
Incorrect mask after softmax, row sum: 0.401112


Result: causal rows block later positions; their allowed weights still sum to 1.
The deliberate incorrect version fails normalization. Future-token isolation
checks information flow, beyond checking that a matrix looks triangular.

My prediction and explanation: _write here_.


## 2. Change values while keeping queries and keys fixed

Predict whether scores, weights, or outputs change.


In [3]:
scores, weights, outputs = attention(X, X, X)
changed_values = [[1.0, 0.0], [0.0, 1.0], [10.0, 10.0]]
new_scores, new_weights, new_outputs = attention(X, X, changed_values)
assert new_scores == scores and new_weights == weights
assert new_outputs != outputs
print('Scores unchanged:', new_scores == scores)
print('Weights unchanged:', new_weights == weights)
print_matrix('Original outputs:', outputs)
print_matrix('Outputs with changed values:', new_outputs)


Scores unchanged: True
Weights unchanged: True
Original outputs:
  [0.802224, 0.598888]
  [0.598888, 0.802224]
  [0.751745, 0.751745]
Outputs with changed values:
  [4.412233, 4.208897]
  [4.208897, 4.412233]
  [5.283154, 5.283154]


Result: scores and weights are unchanged. The weighted content changes.
This separates routing information in Q/K from content in V.

My prediction and explanation: _write here_.


## 3. Scaling and concentration

Use controlled scores [-8, 0, 8], with d_k = 64. This illustrates the
effect of scaling a fixed score row; it is not a benchmark of trained vectors.
Entropy here is -sum(p * log(p)), in nats. A larger entropy means a more
spread-out distribution. Predict which version has larger entropy.


In [4]:
raw_scores = [-8.0, 0.0, 8.0]
scaled_scores = [score / sqrt(64) for score in raw_scores]
raw_weights, scaled_weights = softmax(raw_scores), softmax(scaled_scores)
def entropy(weights):
    return -sum(p * log(p) for p in weights if p > 0)
print('Without scaling:', [round(p, 6) for p in raw_weights])
print('With scaling:   ', [round(p, 6) for p in scaled_weights])
print('Entropy without scaling:', round(entropy(raw_weights), 6))
print('Entropy with scaling:   ', round(entropy(scaled_weights), 6))
assert entropy(scaled_weights) > entropy(raw_weights)
if plt is not None:
    positions = range(3)
    plt.figure(figsize=(6, 3))
    plt.bar([j - 0.2 for j in positions], raw_weights, width=0.4, label='Raw scores')
    plt.bar([j + 0.2 for j in positions], scaled_weights, width=0.4, label='Scaled scores')
    plt.xticks(list(positions), tokens)
    plt.ylabel('Attention weight')
    plt.legend()
    plt.tight_layout()
    plt.show()


Without scaling: [0.0, 0.000335, 0.999665]
With scaling:    [0.090031, 0.244728, 0.665241]
Entropy without scaling: 0.00302
Entropy with scaling:    0.832396


Result: scaling reduces concentration in this example. It does not force
all weights to be equal. Try changing the score separation to test the limit.

My prediction and explanation: _write here_.


## 4. Separate projections

The matrices below are fixed examples. A trained model would learn them.
Predict Q, K, V, and the first causal output before running.


In [5]:
W_Q = [[1.0, 1.0], [0.0, 1.0]]
W_K = [[1.0, 0.0], [0.0, 1.0]]
W_V = [[2.0, 0.0], [0.0, 0.5]]
Q, K, V = project(X, W_Q), project(X, W_K), project(X, W_V)
print_matrix('Q:', Q)
print_matrix('K:', K)
print_matrix('V:', V)
_, projected_weights, projected_outputs = attention(Q, K, V, causal=True)
print_matrix('Projected causal weights:', projected_weights)
print_matrix('Projected causal outputs:', projected_outputs)
assert projected_outputs[0] == V[0]
three_dimensional_values = project(X, [[1.0, 0.0, 1.0], [0.0, 1.0, -1.0]])
assert len(attention(Q, K, three_dimensional_values)[2][0]) == 3
print('Value width can differ from query/key width: three output coordinates checked.')


Q:
  [1.000000, 1.000000]
  [0.000000, 1.000000]
  [1.000000, 2.000000]
K:
  [1.000000, 0.000000]
  [0.000000, 1.000000]
  [1.000000, 1.000000]
V:
  [2.000000, 0.000000]
  [0.000000, 0.500000]
  [2.000000, 0.500000]
Projected causal weights:
  [1.000000, 0.000000, 0.000000]
  [0.330238, 0.669762, 0.000000]
  [0.140029, 0.283995, 0.575975]
Projected causal outputs:
  [2.000000, 0.000000]
  [0.660477, 0.334881]
  [1.432009, 0.429985]
Value width can differ from query/key width: three output coordinates checked.


Result: projections can give queries, keys, and values distinct roles.
The first causal output is still the first value, regardless of routing scores.

My prediction and explanation: _write here_.

## Follow-up

Rebuild the attention function from memory. Next, add two heads with distinct
projections and an output projection. Write the shapes before the loops.
